# 04 · Baseline 1: direct LLM

The simplest method: paste the **entire context** into one prompt and ask the question.
It is what "just use a long-context model" means, and it is what context rot hurts.

On the dev model (qwen3:8b on CPU) this is a **smoke test**: one question per tier at 32K,
to prove the code path works. The real numbers come from Claude in Phase 5.

In [1]:
import json
from pathlib import Path

import pandas as pd
from rlm_logistics.llm import LLM
from rlm_logistics import methods

ROOT = Path("..")
DEV = ROOT / "results/dev"          # dev runs are not committed; final runs (Phase 5) go to results/runs
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
qs = pd.read_json(ROOT / "data/processed/questions.jsonl", lines=True)
lines = (ROOT / "data/processed/context_2m.txt").read_text(encoding="utf-8").split("\n")
sample = qs[qs["size"] == "32k"].groupby("tier").head(1)   # one question per tier: a smoke test
sample[["id", "question"]]

,id,question
0,32k-T1-0,What is the main complaint in the review of or...
6,32k-T2-0,Among orders delivered after the estimated dat...
12,32k-T3-0,"Among all orders, list every pair of orders fr..."


## The prompt
Every method shares one system prompt (`methods.SYSTEM`): the record format, the same label definitions used to build the
answer key, and the rule *end with `ANSWER: ...`*, so `eval.score` knows where the answer is.

In [2]:
print(methods.SYSTEM)

You answer questions about Brazilian e-commerce orders. Each record is one line:
[order ID] seller=ID (state/city) customer=state/city category=... purchased/estimated/delivered dates (delivery status) review=score/5 "comment in Portuguese"
Questions about complaints use these review labels:
Labels (code: meaning):
- d (delivery): complaint about late, slow, lost or not-received delivery, or damage in transit
- p (product): complaint about quality, defects, wrong size/colour, not as described, product broke
- s (seller_service): complaint about the seller: communication, refund, cancellation, wrong or missing items sent, invoice
- g (positive): praise or satisfaction with no real complaint, even if it mentions fast delivery or a good product
- o (other): anything else, or too vague to tell
Rules:
1. Never arrived / not received -> d, even if the seller also did not reply.
2. Wrong item, missing items or wrong quantity sent -> s.
3. Arrived broken or defective -> p.
4. Neutral but satis

## Running it
qwen3:8b can read about 40K tokens, so 32K is the only size it can attempt directly; larger contexts simply do not fit.
On CPU, reading ~28K tokens takes ~30 minutes per question, and writing a long T3 pair list took over an hour,
so the smoke test covers T1 and T2. `methods.run` reuses saved runs, so re-running this notebook is instant.

In [3]:
llm = LLM("ollama", "qwen3:8b", num_ctx=40960)
context = "\n".join(lines[:meta["32k"]["n_records"]])
rows = [methods.run("direct", q, lambda: methods.direct(q["question"], llm, context), llm.model, DEV)
        for q in sample[sample.tier != "T3"].to_dict("records")]
pd.DataFrame(rows)[["id", "score", "input_tokens", "output_tokens", "seconds"]]

,id,score,input_tokens,output_tokens,seconds
0,32k-T1-0,0.0,27956,5,1852.7
1,32k-T2-0,0.0,27960,272,1743.5


In [4]:
for r in rows:
    print(r["id"], "->", r["reply"][-300:], "\n")

32k-T1-0 -> ANSWER: other 

32k-T2-0 -> tes a **delivery complaint** (late delivery), so it is labeled as **d**.

### Step 3: Calculate the percentage
Since there is only one order delivered after the estimated date, and it is a delivery complaint, the percentage is:

$$
\frac{1}{1} \times 100 = 100.0\%
$$

### Final Answer:
ANSWER: 100.0 



## What to notice
* `input_tokens` is ~32K for every question, even the T1 lookup that needs one line: direct LLM always pays for everything.
* Whether qwen gets these right says little (8B model, 3 questions). What matters here is that a reply comes back, gets scored and is logged.